# MCGS Stability Analysis

Loads runs from the `mcgs-stability` MLflow experiment (10 seeds Ã— 30 images) and produces a summary table with:
- **Mean intra-image IoU Â± std** â€” pairwise IoU of region_0 segments across seeds, averaged over images
- **Mean intra-image CV of log-odds drop Â± std** â€” coefficient of variation of log-odds drop across seeds per image, averaged over images
- **Mean intra-image CV of probability drop Â± std** â€” same for probability drop

In [1]:
import itertools
import os

import mlflow
import numpy as np
import pandas as pd
from tqdm.auto import tqdm


MLFLOW_URI = "https://mlflow.rationai.cloud.e-infra.cz/"
EXPERIMENT_NAME = "ucb-stability-recommended"

os.environ["MLFLOW_TRACKING_USERNAME"] = "YOUR_MLFLOW_USERNAME"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "YOUR_MLFLOW_PASSWORD"

mlflow.set_tracking_uri(MLFLOW_URI)
client = mlflow.MlflowClient()

## Load runs

In [2]:
exp = client.get_experiment_by_name(EXPERIMENT_NAME)
if exp is None:
    raise RuntimeError(f"Experiment not found: {EXPERIMENT_NAME}")

all_runs = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    max_results=10000,
    output_format="pandas",
)
print(f"Loaded {len(all_runs)} runs total")

PARENT_TAG = "tags.mlflow.parentRunId"
child_runs = all_runs[all_runs[PARENT_TAG].notna()].copy()
parent_runs = all_runs[all_runs[PARENT_TAG].isna()].copy()
print(f"Parent runs: {len(parent_runs)}, child (per-image) runs: {len(child_runs)}")

Loaded 930 runs total
Parent runs: 30, child (per-image) runs: 900


In [3]:
# Propagate params from parent to children
parent_param_cols = sorted(c for c in parent_runs.columns if c.startswith("params."))
overlap = [c for c in parent_param_cols if c in child_runs.columns]
child_runs = child_runs.drop(columns=overlap)
parent_params = parent_runs.set_index("run_id")[parent_param_cols]
child_runs = child_runs.join(parent_params, on=PARENT_TAG)

for col in parent_param_cols:
    if col in child_runs.columns:
        child_runs[col] = pd.to_numeric(child_runs[col], errors="coerce")

desired_col = "params.desired_length"
if desired_col in child_runs.columns:
    unique_lengths = sorted(child_runs[desired_col].dropna().unique().tolist())
    print(f"Unique desired_lengths in experiment: {unique_lengths}")
else:
    print(f"Warning: {desired_col} not found")

METRIC_COLS = [
    "metrics.region_0/probability_drop",
    "metrics.region_0/original_prob",
    "metrics.region_0/masked_prob",
    "metrics.original_log_odds",
]
for col in METRIC_COLS:
    if col in child_runs.columns:
        child_runs[col] = pd.to_numeric(child_runs[col], errors="coerce")

child_runs = child_runs.rename(
    columns={
        "metrics.region_0/probability_drop": "prob_drop",
        "metrics.region_0/original_prob": "original_prob",
        "metrics.region_0/masked_prob": "masked_prob",
        "metrics.original_log_odds": "original_log_odds",
        "params.seed": "seed",
        "params.desired_length": "desired_length",
        "tags.mlflow.runName": "image_name",
    },
)

# log-odds drop = original_log_odds - masked_log_odds
# masked_log_odds = log(masked_prob / (1 - masked_prob))
eps = 1e-7
child_runs["masked_log_odds"] = np.log(
    (child_runs["masked_prob"] + eps) / (1 - child_runs["masked_prob"] + eps)
)
child_runs["log_odds_drop"] = (
    child_runs["original_log_odds"] - child_runs["masked_log_odds"]
)

print(
    child_runs[
        ["image_name", "desired_length", "seed", "prob_drop", "log_odds_drop"]
    ].head(10)
)

Unique desired_lengths in experiment: [15, 30, 90]
           image_name  desired_length  seed  prob_drop  log_odds_drop
0  imagenets_0026.jpg              90    10   0.015670       5.570225
1  imagenets_0025.jpg              90    10   0.698309      11.481048
2  imagenets_0017.jpg              90    10   0.982883      10.848834
3  imagenets_0029.jpg              90    10   0.880977       7.453302
4  imagenets_0004.jpg              90    10   0.987252      12.554431
5  imagenets_0014.jpg              90    10   0.938421      18.841368
6  imagenets_0011.jpg              90    10   0.289359       4.300247
7  imagenets_0018.jpg              90    10   0.987546      19.123157
8  imagenets_0012.jpg              90    10   0.498268      16.102124
9  imagenets_0022.jpg              90    10   0.897524       9.278424


## Load region_0 segments from MLflow artifacts

In [4]:
import json
import tempfile


def load_segments(run_id: str) -> set[int] | None:
    """Download region_0/segments.json for a run and return the segment set."""
    try:
        with tempfile.TemporaryDirectory() as tmp:
            path = mlflow.artifacts.download_artifacts(
                run_id=run_id,
                artifact_path="region_0/segments.json",
                dst_path=tmp,
            )
            with open(path) as f:
                data = json.load(f)
            return set(data["segments"])
    except Exception:
        return None


print(f"Loading segments for {len(child_runs)} child runs...")
segments_map: dict[str, set[int]] = {}
for run_id in tqdm(child_runs["run_id"].tolist()):
    segs = load_segments(run_id)
    if segs is not None:
        segments_map[run_id] = segs

print(f"Successfully loaded segments for {len(segments_map)} runs")
child_runs["segments"] = child_runs["run_id"].map(segments_map)

Loading segments for 900 child runs...


  0%|          | 0/900 [00:00<?, ?it/s]

Successfully loaded segments for 900 runs


## Compute intra-image IoU across seeds

In [5]:
def pairwise_iou(segment_sets: list[set[int]]) -> float:
    """Mean pairwise IoU over all pairs in the list."""
    ious = []
    for a, b in itertools.combinations(segment_sets, 2):
        inter = len(a & b)
        union = len(a | b)
        ious.append(inter / union if union > 0 else 1.0)
    return float(np.mean(ious)) if ious else float("nan")


valid = child_runs.dropna(subset=["segments"])
iou_per_image = (
    valid.groupby(["desired_length", "image_name"])["segments"]
    .apply(lambda s: pairwise_iou(s.tolist()))
    .rename("mean_pairwise_iou")
    .reset_index()
)

print(iou_per_image)

    desired_length          image_name  mean_pairwise_iou
0               15  imagenets_0000.jpg           0.423936
1               15  imagenets_0001.jpg           0.666267
2               15  imagenets_0002.jpg           0.500057
3               15  imagenets_0003.jpg           0.483611
4               15  imagenets_0004.jpg           0.615111
..             ...                 ...                ...
85              90  imagenets_0025.jpg           0.445126
86              90  imagenets_0026.jpg           0.413631
87              90  imagenets_0027.jpg           0.205661
88              90  imagenets_0028.jpg           0.421574
89              90  imagenets_0029.jpg           0.383096

[90 rows x 3 columns]


## Compute intra-image CV of probability drop and log-odds drop

In [6]:
def cv(series: pd.Series) -> float:
    """Coefficient of variation = std / mean. Returns NaN if mean ~ 0."""
    m = series.mean()
    s = series.std()
    return float(s / m) if abs(m) > 1e-9 else float("nan")


per_image_cv = (
    child_runs.groupby(["desired_length", "image_name"])
    .agg(
        cv_prob_drop=("prob_drop", cv),
        cv_log_odds_drop=("log_odds_drop", cv),
        n_seeds=("seed", "count"),
    )
    .reset_index()
)

print(per_image_cv)

    desired_length          image_name  cv_prob_drop  cv_log_odds_drop  \
0               15  imagenets_0000.jpg  1.916157e-01      5.837346e-02   
1               15  imagenets_0001.jpg  7.389427e-02      2.679667e-02   
2               15  imagenets_0002.jpg  4.724904e-02      2.794030e-02   
3               15  imagenets_0003.jpg  2.062530e-02      4.236566e-02   
4               15  imagenets_0004.jpg  3.151148e-01      7.102542e-02   
..             ...                 ...           ...               ...   
85              90  imagenets_0025.jpg  1.070185e-05      1.045658e-01   
86              90  imagenets_0026.jpg  6.135562e-01      1.125228e-01   
87              90  imagenets_0027.jpg  1.856300e-02      9.640679e-02   
88              90  imagenets_0028.jpg  2.017582e-12      5.627738e-07   
89              90  imagenets_0029.jpg  1.968147e-03      6.075126e-02   

    n_seeds  
0        10  
1        10  
2        10  
3        10  
4        10  
..      ...  
85       10  

## Summary table

In [7]:
summary = per_image_cv.merge(
    iou_per_image, on=["desired_length", "image_name"], how="left"
)

agg = summary.groupby("desired_length").agg(
    iou_mean=("mean_pairwise_iou", "mean"),
    iou_std=("mean_pairwise_iou", "std"),
    cv_log_mean=("cv_log_odds_drop", "mean"),
    cv_log_std=("cv_log_odds_drop", "std"),
    cv_prob_mean=("cv_prob_drop", "mean"),
    cv_prob_std=("cv_prob_drop", "std"),
)


def fmt(m, s):
    return f"{m:.4f} Â± {s:.4f}"


rows = {
    "Intra-image IoU": agg.apply(lambda r: fmt(r.iou_mean, r.iou_std), axis=1),
    "CV(log-odds drop)": agg.apply(lambda r: fmt(r.cv_log_mean, r.cv_log_std), axis=1),
    "CV(prob drop)": agg.apply(lambda r: fmt(r.cv_prob_mean, r.cv_prob_std), axis=1),
}

table = pd.DataFrame(rows).T
table.columns = [f"L={int(l)}" for l in table.columns]
table.index.name = "Metric"
print(table.to_string())
table

                              L=15             L=30             L=90
Metric                                                              
Intra-image IoU    0.5527 Â± 0.1254  0.4781 Â± 0.0929  0.3489 Â± 0.0788
CV(log-odds drop)  0.0462 Â± 0.0330  0.0691 Â± 0.0390  0.0976 Â± 0.0797
CV(prob drop)      0.0574 Â± 0.0770  0.0749 Â± 0.1073  0.0537 Â± 0.1559


,L=15,L=30,L=90
Metric,,,
Intra-image IoU,0.5527 Â± 0.1254,0.4781 Â± 0.0929,0.3489 Â± 0.0788
CV(log-odds drop),0.0462 Â± 0.0330,0.0691 Â± 0.0390,0.0976 Â± 0.0797
CV(prob drop),0.0574 Â± 0.0770,0.0749 Â± 0.1073,0.0537 Â± 0.1559


In [8]:
# Per-image breakdown
print(summary.to_string(index=False))
summary

 desired_length         image_name  cv_prob_drop  cv_log_odds_drop  n_seeds  mean_pairwise_iou
             15 imagenets_0000.jpg  1.916157e-01      5.837346e-02       10           0.423936
             15 imagenets_0001.jpg  7.389427e-02      2.679667e-02       10           0.666267
             15 imagenets_0002.jpg  4.724904e-02      2.794030e-02       10           0.500057
             15 imagenets_0003.jpg  2.062530e-02      4.236566e-02       10           0.483611
             15 imagenets_0004.jpg  3.151148e-01      7.102542e-02       10           0.615111
             15 imagenets_0005.jpg  8.036011e-03      1.089332e-01       10           0.539510
             15 imagenets_0006.jpg  1.475063e-01      6.260854e-02       10           0.301595
             15 imagenets_0007.jpg  4.372468e-02      7.340089e-02       10           0.673520
             15 imagenets_0008.jpg  1.456062e-02      1.437303e-01       10           0.680465
             15 imagenets_0009.jpg  9.453782e-02  

,desired_length,image_name,cv_prob_drop,cv_log_odds_drop,n_seeds,mean_pairwise_iou
0,15,imagenets_0000.jpg,1.916157e-01,5.837346e-02,10,0.423936
1,15,imagenets_0001.jpg,7.389427e-02,2.679667e-02,10,0.666267
2,15,imagenets_0002.jpg,4.724904e-02,2.794030e-02,10,0.500057
3,15,imagenets_0003.jpg,2.062530e-02,4.236566e-02,10,0.483611
4,15,imagenets_0004.jpg,3.151148e-01,7.102542e-02,10,0.615111
...,...,...,...,...,...,...
85,90,imagenets_0025.jpg,1.070185e-05,1.045658e-01,10,0.445126
86,90,imagenets_0026.jpg,6.135562e-01,1.125228e-01,10,0.413631
87,90,imagenets_0027.jpg,1.856300e-02,9.640679e-02,10,0.205661
88,90,imagenets_0028.jpg,2.017582e-12,5.627738e-07,10,0.421574
